# NB08 — Cosine-Similarity Cross-Check of the Specimen Split

### Why this notebook exists

The specimen definition used throughout this project (pHash clustering, Hamming ≤ 5,
plus bounded 30 s temporal blocks) has one acknowledged limitation, stated directly in
the manuscript's Methods section:

> "The pHash cluster is necessary but not sufficient as a grouping unit: it catches
> burst-shot near-duplicates but not two separate photographs of the same leaf taken
> moments apart from different angles."

And in the project's internal run notes (`RUN_ORDER.md`):

> "You cannot measure grouping purity on real data — there are no ground-truth leaf IDs."

Cosine similarity on deep CNN feature embeddings is a standard way to probe that gap
directly rather than leave it unquantified. Two photos of the same physical leaf from
slightly different angles will usually still point in nearly the same direction in a
good visual-feature space, even when their pHash bits differ enough to land in
different pHash clusters (and therefore, potentially, different specimens). This
notebook tests that directly instead of assuming it.

### What it does

1. Extracts a deep feature embedding for every one of the 11,094 raw images using an
   **ImageNet-pretrained ResNet-50 that is NOT fine-tuned on BDLitchi.** This is
   deliberate: a classifier fine-tuned on the 11 disease labels would encode *class
   identity* in its embedding, so two different leaves of the same disease could look
   "similar" for the wrong reason. A generic pretrained backbone instead encodes
   general visual appearance, which is what a duplicate-photo check needs.
2. For every one of the 26 capture sessions, computes the full pairwise cosine
   similarity between every image in that session (specimens never span sessions, so
   cross-session comparisons are not meaningful here).
3. Splits every pair into two groups: **same-specimen** pairs (already agreed, by the
   current pHash+temporal-block rule, to be the same physical leaf) and
   **cross-specimen** pairs (currently treated as independent).
4. Flags cross-specimen pairs whose cosine similarity is as high as a typical
   same-specimen pair — these are the candidates the current specimen rule may be
   *missing*: two images that look like the same leaf but were split apart. It also
   flags unusually low-similarity same-specimen pairs, as a check in the other
   direction (possible *over*-merging).
5. Saves a flagged-pairs CSV, a histogram figure, a LaTeX summary table, and a small
   visual contact sheet of the most suspicious pairs for direct inspection.

### Before running

- Attach the raw BDLitchi image dataset (the 11 class folders) and set `RAW_DATASET_DIR`.
- Attach the NB01 output dataset (`bdlitchi-revision-splits`, contains
  `manifest_full.csv`) and set `SPLITS_DIR`.
- GPU is optional. This is 11,094 forward passes through a frozen ResNet-50 with no
  training — a few minutes on a T4, well under ten minutes on CPU.
- Nothing here needs any of NB02–NB06's checkpoints. This notebook is fully independent
  of them.

### After running

`revision_cosine/manuscript_numbers_cosine.md` and the flagged-pairs contact sheet are
the outputs of interest. Two outcomes are both informative, not just one:
- **Few or no high-similarity cross-specimen pairs** — positive evidence the specimen
  construction is sound, worth stating explicitly in the paper rather than leaving as an
  unquantified limitation.
- **A meaningful number of flagged pairs** — a genuine residual-leakage finding, in the
  same spirit as the rest of the audit: report the count and give one or two example pairs.


In [ ]:
# ===== Imports =====
import os, json, time, random, warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# ===== Config =====
RAW_DATASET_DIR = '/kaggle/input/datasets/maruf170102/bdlithi/Dataset'   # <-- EDIT: folder containing the 11 class sub-folders
SPLITS_DIR = Path('/kaggle/input/datasets/maruf170102/bdlitchi-revision-splits/revision_splits')  # <-- EDIT: NB01 output

OUT_DIR = Path('/kaggle/working/revision_cosine'); OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 64
NUM_WORKERS = 2
# Percentile of the same-specimen similarity distribution used as the flagging
# threshold for cross-specimen pairs (see Section 5 below for why this, not a
# fixed constant like 0.95, is the right way to set it).
FLAG_PERCENTILE = 5
FLAG_PERCENTILES_SWEEP = [1, 2, 5, 10]   # for the threshold-sensitivity table
TOP_K_CONTACT_SHEET = 12   # how many flagged pairs to render as thumbnails

print('RAW_DATASET_DIR exists:', os.path.isdir(RAW_DATASET_DIR))
print('SPLITS_DIR exists:', SPLITS_DIR.exists())


In [ ]:
# ===== Load manifest, rebuild local paths, verify every file resolves =====
# Same defensive pattern as NB02-NB05: fail loudly before any compute starts,
# not several minutes into a DataLoader worker.
manifest = pd.read_csv(SPLITS_DIR / 'manifest_full.csv')
manifest['local_path'] = manifest.apply(
    lambda r: os.path.join(RAW_DATASET_DIR, r['label'], r['filename']), axis=1
)
missing = manifest.loc[~manifest['local_path'].apply(os.path.exists)]
if len(missing):
    print(missing.head(10))
    raise FileNotFoundError(f'{len(missing)} of {len(manifest)} image paths do not resolve. '
                             f'Fix RAW_DATASET_DIR before continuing.')
print(f'OK - all {len(manifest)} image paths resolve in this session.')
print(manifest[['session_id', 'cluster_id', 'specimen_id']].nunique())


In [ ]:
# ===== Feature extractor: ImageNet-pretrained ResNet-50, NOT fine-tuned =====
IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths; self.t = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.t(img), i

backbone = torchvision.models.resnet50(weights='IMAGENET1K_V2')
backbone.fc = nn.Identity()   # expose the 2048-d pooled penultimate feature
backbone = backbone.to(device).eval()
for p in backbone.parameters():
    p.requires_grad_(False)

print('Feature extractor ready: ResNet-50, ImageNet-pretrained, 2048-d embeddings.')


In [ ]:
# ===== Extract an embedding for every image =====
paths = manifest['local_path'].tolist()
ds = ImagePathDataset(paths, eval_tf)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

embeddings = np.zeros((len(paths), 2048), dtype=np.float32)
t0 = time.time()
with torch.no_grad():
    for bi, (x, idx) in enumerate(dl):
        x = x.to(device, non_blocking=True)
        feat = backbone(x).cpu().numpy()
        embeddings[idx.numpy()] = feat
        if (bi + 1) % 20 == 0 or (bi + 1) == len(dl):
            print(f'batch {bi+1}/{len(dl)}  elapsed={time.time()-t0:.1f}s')

np.save(OUT_DIR / 'embeddings.npy', embeddings)
manifest[['filename', 'label', 'session_id', 'cluster_id', 'specimen_id']].to_csv(
    OUT_DIR / 'embeddings_meta.csv', index=False
)
print('Done in', round(time.time() - t0, 1), 's. Embeddings shape:', embeddings.shape)


In [ ]:
# ===== Pairwise cosine similarity, within each session, same- vs cross-specimen =====
# L2-normalise once; cosine similarity between two normalised vectors is just their dot product.
norm = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)

same_specimen_sims = []
same_specimen_records = []    # (session_id, idx_a, idx_b, cosine_sim)
cross_specimen_records = []   # (session_id, idx_a, idx_b, specimen_a, specimen_b, cosine_sim)

manifest_reset = manifest.reset_index(drop=True)
for sess_id, group in manifest_reset.groupby('session_id'):
    idx = group.index.to_numpy()
    if len(idx) < 2:
        continue
    sub = norm[idx]                       # (n_sess, 2048)
    sim = sub @ sub.T                     # (n_sess, n_sess) cosine similarity
    specimens = group['specimen_id'].to_numpy()
    n = len(idx)
    iu, ju = np.triu_indices(n, k=1)      # upper triangle, no self-pairs, no double-count
    sims = sim[iu, ju]
    a_idx, b_idx = idx[iu], idx[ju]
    spec_a, spec_b = specimens[iu], specimens[ju]
    same_mask = spec_a == spec_b          # this is the comparison against your existing
                                           # pHash + temporal-block specimen clustering

    same_specimen_sims.append(sims[same_mask])
    same_specimen_records.extend(zip([sess_id] * same_mask.sum(), a_idx[same_mask], b_idx[same_mask], sims[same_mask]))
    cross_specimen_records.extend(zip([sess_id] * (~same_mask).sum(), a_idx[~same_mask], b_idx[~same_mask],
                                       spec_a[~same_mask], spec_b[~same_mask], sims[~same_mask]))

same_sim_all = np.concatenate(same_specimen_sims) if same_specimen_sims else np.array([])
same_df = pd.DataFrame(same_specimen_records, columns=['session_id', 'idx_a', 'idx_b', 'cosine_sim'])
cross_df = pd.DataFrame(cross_specimen_records,
                         columns=['session_id', 'idx_a', 'idx_b', 'specimen_a', 'specimen_b', 'cosine_sim'])

print('same-specimen pairs:', len(same_df), ' cross-specimen pairs:', len(cross_df))
print('same-specimen cosine sim: mean=%.4f median=%.4f p5=%.4f' %
      (same_sim_all.mean(), np.median(same_sim_all), np.percentile(same_sim_all, FLAG_PERCENTILE)))
print('cross-specimen cosine sim: mean=%.4f median=%.4f p95=%.4f' %
      (cross_df.cosine_sim.mean(), cross_df.cosine_sim.median(), np.percentile(cross_df.cosine_sim, 100-FLAG_PERCENTILE)))


In [ ]:
# ===== Flagging =====
# Threshold = the p-th percentile of the SAME-specimen similarity distribution.
# Rationale: rather than pick an arbitrary constant like 0.95, use the data's own
# notion of "how similar do we already agree same-leaf photos are" and ask which
# cross-specimen pairs clear that same bar.
threshold_high = float(np.percentile(same_sim_all, FLAG_PERCENTILE))
flagged_cross = cross_df[cross_df.cosine_sim >= threshold_high].sort_values('cosine_sim', ascending=False)

# Reverse check: same-specimen pairs that look surprisingly LESS similar than most
# cross-specimen pairs -- a possible over-merging symptom (lower priority: the
# project's own design notes call over-merging "the safe error", but worth reporting).
threshold_low = float(np.percentile(cross_df.cosine_sim, 100 - FLAG_PERCENTILE))
flagged_same_low = same_df[same_df.cosine_sim <= threshold_low].sort_values('cosine_sim')

print(f'Flagging threshold (same-specimen p{FLAG_PERCENTILE}): {threshold_high:.4f}')
print(f'Cross-specimen pairs at/above that threshold: {len(flagged_cross)} of {len(cross_df)} '
      f'({100*len(flagged_cross)/max(len(cross_df),1):.3f}%)')
print(f'Distinct specimens touched: {pd.unique(flagged_cross[["specimen_a","specimen_b"]].values.ravel()).shape[0]}')
print()
print(f'Reverse check threshold (cross-specimen p{100-FLAG_PERCENTILE}): {threshold_low:.4f}')
print(f'Same-specimen pairs at/below that threshold: {len(flagged_same_low)} of {len(same_df)}')

# attach readable metadata (filenames, labels) for manual inspection
meta_lookup = manifest_reset[['filename', 'label', 'specimen_id']]
flagged_cross = flagged_cross.merge(meta_lookup, left_on='idx_a', right_index=True) \
                              .rename(columns={'filename': 'filename_a', 'label': 'label_a'}) \
                              .drop(columns='specimen_id') \
                              .merge(meta_lookup, left_on='idx_b', right_index=True) \
                              .rename(columns={'filename': 'filename_b', 'label': 'label_b'}) \
                              .drop(columns='specimen_id')

flagged_cross.to_csv(OUT_DIR / 'flagged_cross_specimen_pairs.csv', index=False)
flagged_same_low.to_csv(OUT_DIR / 'flagged_same_specimen_low_similarity.csv', index=False)
print('\nsaved flagged_cross_specimen_pairs.csv and flagged_same_specimen_low_similarity.csv')
flagged_cross.head(10)


In [ ]:
# ===== Plot style + save_fig helper (defined once, used by every figure below) =====
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.alpha': 0.22, 'grid.linewidth': 0.5, 'axes.axisbelow': True,
    'axes.edgecolor': '#666666', 'axes.linewidth': 0.8,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 10,
    'legend.frameon': False, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')


In [ ]:
# ===== pHash Hamming-distance diagnostic =====
# Ties this check directly back to the method already used everywhere else in the
# project (Section "Near-Duplicate Removal": 64-bit pHash, Union-Find, Hamming <= 5).
# For every cross-specimen pair, this answers: was a flagged pair a near-miss just
# above the Hamming <= 5 clustering cutoff, or a genuine blind spot (very different
# pHash, high visual similarity -- exactly the "same leaf, different angle" case pHash
# alone cannot catch)? Those two situations argue for different text in the paper.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imagehash'], check=True)
import imagehash

t0 = time.time()
phash_ints = np.zeros(len(paths), dtype=np.uint64)  # 64-bit hash can exceed signed int64 range
for i, p in enumerate(paths):
    h = imagehash.phash(Image.open(p).convert('RGB'), hash_size=8)   # 64-bit hash, matches Methods
    phash_ints[i] = int(str(h), 16)
    if (i + 1) % 2000 == 0:
        print(f'phash {i+1}/{len(paths)}  elapsed={time.time()-t0:.1f}s')
print('pHash recomputation done in', round(time.time() - t0, 1), 's')

def hamming64(a, b):
    x = np.bitwise_xor(a.astype(np.uint64), b.astype(np.uint64))
    return np.array([bin(int(v)).count('1') for v in x])

cross_df['phash_hamming'] = hamming64(phash_ints[cross_df.idx_a.to_numpy()], phash_ints[cross_df.idx_b.to_numpy()])
same_df['phash_hamming'] = hamming64(phash_ints[same_df.idx_a.to_numpy()], phash_ints[same_df.idx_b.to_numpy()])
flagged_cross['phash_hamming'] = hamming64(phash_ints[flagged_cross.idx_a.to_numpy()], phash_ints[flagged_cross.idx_b.to_numpy()])

PHASH_CLUSTER_CUTOFF = 5   # the Hamming threshold already used for pHash clustering elsewhere in this project
near_miss = flagged_cross[flagged_cross.phash_hamming <= PHASH_CLUSTER_CUTOFF + 3]
genuine_blindspot = flagged_cross[flagged_cross.phash_hamming > PHASH_CLUSTER_CUTOFF + 3]
print(f"Flagged pairs within 3 of the pHash cutoff (near-miss, Hamming <= {PHASH_CLUSTER_CUTOFF+3}): {len(near_miss)}")
print(f"Flagged pairs clearly outside pHash range (Hamming > {PHASH_CLUSTER_CUTOFF+3}), i.e. genuine pHash blind spots: {len(genuine_blindspot)}")
print('Mean pHash Hamming distance -- same-specimen:', same_df.phash_hamming.mean(),
      ' cross-specimen (all):', cross_df.phash_hamming.mean(),
      ' cross-specimen (flagged):', flagged_cross.phash_hamming.mean() if len(flagged_cross) else float('nan'))


In [ ]:
# ===== Figure: cosine similarity vs. pHash Hamming distance (cross-specimen pairs) =====
# A random subsample keeps the scatter readable and the notebook fast; the full
# distribution stats above already use every pair.
rng = np.random.default_rng(SEED)
sample_n = min(20000, len(cross_df))
sample = cross_df.sample(sample_n, random_state=SEED) if len(cross_df) > sample_n else cross_df

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(sample.phash_hamming, sample.cosine_sim, s=4, alpha=0.15, color='#8FAFD4', label='cross-specimen pair')
if len(flagged_cross):
    ax.scatter(flagged_cross.phash_hamming, flagged_cross.cosine_sim, s=14, alpha=0.85,
               color='#C44E52', label='flagged (candidate missed duplicate)')
ax.axhline(threshold_high, color='#666666', linestyle=':', linewidth=1, label=f'flagging threshold ({threshold_high:.3f})')
ax.axvline(PHASH_CLUSTER_CUTOFF, color='#333333', linestyle='--', linewidth=1, label=f'pHash cluster cutoff (Hamming={PHASH_CLUSTER_CUTOFF})')
ax.set_xlabel('pHash Hamming distance')
ax.set_ylabel('Cosine similarity (ResNet-50 embeddings)')
ax.set_title('Cross-specimen pairs: cosine similarity vs. pHash distance')
ax.legend(fontsize=8, loc='lower left')
save_fig(fig, 'fig_cosine_vs_phash_hamming')
plt.show()


In [ ]:
# ===== Threshold-sensitivity table =====
# Same logic the project already applies to SPECIMEN_BLOCK_SECONDS: don't assert a
# single percentile, show what changes at a few others so the choice isn't arbitrary.
sens_rows = []
for pct in FLAG_PERCENTILES_SWEEP:
    thr = float(np.percentile(same_sim_all, pct))
    n_flag = int((cross_df.cosine_sim >= thr).sum())
    sens_rows.append({
        'percentile': pct, 'threshold': thr, 'n_flagged': n_flag,
        'pct_of_cross_pairs': 100 * n_flag / max(len(cross_df), 1),
    })
sensitivity_df = pd.DataFrame(sens_rows)
sensitivity_df.to_csv(OUT_DIR / 'threshold_sensitivity.csv', index=False)
print(sensitivity_df.to_string(index=False))

lines = [
    "\\begin{table}[H]",
    "\\centering",
    "\\caption{Sensitivity of the flagged cross-specimen pair count to the choice of "
    "flagging percentile (percentile of the same-specimen cosine-similarity "
    "distribution used as the threshold).}",
    "\\label{tab:cosine_threshold_sensitivity}",
    "\\begin{tabular}{rrrr}",
    "\\toprule",
    "Percentile & Threshold & Flagged pairs & \\% of cross-specimen pairs \\\\",
    "\\midrule",
]
for _, r in sensitivity_df.iterrows():
    lines.append(f"{int(r.percentile)} & {r.threshold:.4f} & {int(r.n_flagged)} & {r.pct_of_cross_pairs:.3f}\\% \\\\")
lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}", ""]
(OUT_DIR / 'table_cosine_threshold_sensitivity.tex').write_text("\n".join(lines))


In [ ]:
# ===== Per-class / per-session breakdown of flagged pairs =====
# Every session in BDLitchi belongs to exactly one class (26 sessions total, matching
# Table "dataset_stats"), so grouping flagged pairs by session's class is well-defined.
# This checks whether flagged pairs concentrate in the four single-session classes
# (Fungal Stripe Damage, Leaf Blight Disease, White Spot, Yellow Mosaic Virus) already
# named in the manuscript as carrying the most cross-split leakage risk.
def latex_escape(s):
    """Escape LaTeX-special characters in data-derived strings (specimen ids, class
    labels) before dropping them into a table. BDLitchi's own naming (e.g. 's000000',
    'Black Spot') doesn't currently need this, but auto-generated LaTeX from data
    should never assume that stays true."""
    s = str(s)
    for a, b in [('\\', r'\textbackslash{}'), ('_', r'\_'), ('%', r'\%'), ('&', r'\&'),
                 ('#', r'\#'), ('$', r'\$'), ('{', r'\{'), ('}', r'\}')]:
        s = s.replace(a, b)
    return s

sess_to_label = manifest_reset.drop_duplicates('session_id').set_index('session_id')['label']

cross_df['label'] = cross_df.session_id.map(sess_to_label)
flagged_cross['label'] = flagged_cross.session_id.map(sess_to_label)

breakdown = pd.DataFrame({
    'cross_specimen_pairs': cross_df.groupby('label').size(),
    'flagged_pairs': flagged_cross.groupby('label').size(),
}).fillna(0)
breakdown['flagged_pairs'] = breakdown['flagged_pairs'].astype(int)
breakdown['pct_flagged'] = 100 * breakdown['flagged_pairs'] / breakdown['cross_specimen_pairs'].replace(0, np.nan)
breakdown = breakdown.sort_values('pct_flagged', ascending=False)
breakdown.to_csv(OUT_DIR / 'flagged_breakdown_by_class.csv')
print(breakdown)

lines = [
    "\\begin{table}[H]",
    "\\centering",
    "\\caption{Flagged cross-specimen pairs by class. Every capture session belongs to "
    "a single class, so this groups directly by session.}",
    "\\label{tab:cosine_flagged_breakdown}",
    "\\begin{tabular}{lrrr}",
    "\\toprule",
    "Class & Cross-specimen pairs & Flagged & \\% flagged \\\\",
    "\\midrule",
]
for cls, r in breakdown.iterrows():
    pct = f"{r.pct_flagged:.3f}\\%" if pd.notna(r.pct_flagged) else "--"
    lines.append(f"{latex_escape(cls)} & {int(r.cross_specimen_pairs)} & {int(r.flagged_pairs)} & {pct} \\\\")
lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}", ""]
(OUT_DIR / 'table_cosine_flagged_breakdown.tex').write_text("\n".join(lines))


In [ ]:
# ===== Figure: same- vs cross-specimen cosine similarity distributions =====
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(cross_df.cosine_sim, bins=80, alpha=0.6, density=True, label=f'cross-specimen (n={len(cross_df):,})', color='#E29497')
ax.hist(same_sim_all, bins=80, alpha=0.6, density=True, label=f'same-specimen (n={len(same_df):,})', color='#8FAFD4')
ax.axvline(threshold_high, color='#C44E52', linestyle='--', linewidth=1.2,
           label=f'flagging threshold ({threshold_high:.3f})')
ax.set_xlabel('Cosine similarity (ResNet-50 ImageNet embeddings)')
ax.set_ylabel('Density')
ax.set_title('Within-session pairwise cosine similarity: same- vs. cross-specimen')
ax.legend()
save_fig(fig, 'fig_cosine_same_vs_cross_specimen')
plt.show()


In [ ]:
# ===== Contact sheet of the most suspicious flagged pairs (visual eyeball check) =====
k = min(TOP_K_CONTACT_SHEET, len(flagged_cross))
if k == 0:
    print('No flagged cross-specimen pairs to display -- nothing to inspect visually.')
else:
    fig, axes = plt.subplots(k, 2, figsize=(4, 2 * k))
    if k == 1:
        axes = axes.reshape(1, 2)
    for row, (_, r) in enumerate(flagged_cross.head(k).iterrows()):
        pa = os.path.join(RAW_DATASET_DIR, r['label_a'], r['filename_a'])
        pb = os.path.join(RAW_DATASET_DIR, r['label_b'], r['filename_b'])
        axes[row, 0].imshow(Image.open(pa).convert('RGB')); axes[row, 0].axis('off')
        axes[row, 1].imshow(Image.open(pb).convert('RGB')); axes[row, 1].axis('off')
        axes[row, 0].set_title(f"{r['specimen_a']} / {r['label_a']}", fontsize=8)
        axes[row, 1].set_title(f"{r['specimen_b']} / {r['label_b']}  sim={r['cosine_sim']:.3f}", fontsize=8)
    fig.suptitle('Most similar cross-specimen pairs (left/right = candidate same leaf?)', y=1.0)
    save_fig(fig, 'fig_cosine_flagged_contact_sheet')
    plt.show()


In [ ]:
# ===== Compact example table (for the manuscript or supplement) =====
K_TABLE = min(8, len(flagged_cross))
ex = flagged_cross.head(K_TABLE)

lines = [
    "\\begin{table}[H]",
    "\\centering",
    "\\small",
    f"\\caption{{The {K_TABLE} most similar flagged cross-specimen pairs: two images "
    "assigned to different specimens by the pHash + temporal-block rule, whose "
    "deep-feature cosine similarity matches typical same-specimen pairs.}",
    "\\label{tab:cosine_example_pairs}",
    "\\begin{tabular}{llrr}",
    "\\toprule",
    "Specimen A & Specimen B & Cosine sim. & pHash Hamming \\\\",
    "\\midrule",
]
for _, r in ex.iterrows():
    lines.append(f"{latex_escape(r.specimen_a)} & {latex_escape(r.specimen_b)} & "
                 f"{r.cosine_sim:.4f} & {int(r.phash_hamming)} \\\\")
lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}", ""]

example_tex = "\n".join(lines)
(OUT_DIR / 'table_cosine_example_pairs.tex').write_text(example_tex)
print(example_tex)


In [ ]:
# ===== LaTeX table + manuscript-numbers summary =====
main_lines = [
    "\\begin{table}[H]",
    "\\centering",
    "\\caption{Cosine-similarity cross-check of the specimen split. For every one of "
    "the 26 capture sessions, pairwise cosine similarity was computed between "
    "ResNet-50 (ImageNet-pretrained, not fine-tuned) embeddings of every image pair "
    "in that session. A cross-specimen pair is flagged when its similarity reaches "
    f"the {FLAG_PERCENTILE}th percentile of the same-specimen similarity "
    "distribution, i.e.\\ it looks as similar as a typical already-agreed same-leaf "
    "pair. See Table~\\ref{tab:cosine_threshold_sensitivity} for sensitivity to that "
    "percentile and Table~\\ref{tab:cosine_flagged_breakdown} for the per-class "
    "breakdown.}",
    "\\label{tab:cosine_specimen_check}",
    "\\begin{tabular}{lr}",
    "\\toprule",
    "Quantity & Value \\\\",
    "\\midrule",
    f"Images embedded & {len(embeddings)} \\\\",
    f"Within-session pairs evaluated & {len(same_df) + len(cross_df)} \\\\",
    f"Same-specimen pairs & {len(same_df)} \\\\",
    f"Cross-specimen pairs & {len(cross_df)} \\\\",
    f"Same-specimen cosine sim.\\ (mean / median) & {same_sim_all.mean():.4f} / {np.median(same_sim_all):.4f} \\\\",
    f"Cross-specimen cosine sim.\\ (mean / median) & {cross_df.cosine_sim.mean():.4f} / {cross_df.cosine_sim.median():.4f} \\\\",
    f"Flagging threshold (same-specimen p{FLAG_PERCENTILE}) & {threshold_high:.4f} \\\\",
    f"Cross-specimen pairs flagged & {len(flagged_cross)} ({100 * len(flagged_cross) / max(len(cross_df), 1):.3f}\\%) \\\\",
    f"Distinct specimens touched by a flagged pair & {pd.unique(flagged_cross[['specimen_a', 'specimen_b']].values.ravel()).shape[0] if len(flagged_cross) else 0} \\\\",
    f"Mean pHash Hamming, same-specimen / cross-specimen / flagged & {same_df.phash_hamming.mean():.1f} / {cross_df.phash_hamming.mean():.1f} / {(flagged_cross.phash_hamming.mean() if len(flagged_cross) else float('nan')):.1f} \\\\",
    f"Flagged pairs within 3 of the pHash cutoff (near-miss) & {len(near_miss)} of {len(flagged_cross)} \\\\",
    "\\bottomrule",
    "\\end{tabular}",
    "\\end{table}",
    "",
]
table_tex = "\n".join(main_lines)
(OUT_DIR / 'table_cosine_specimen_check.tex').write_text(table_tex)

hotspot = breakdown.index[0] if len(breakdown) and breakdown['pct_flagged'].notna().any() else 'n/a'

summary = {
    'backbone': 'resnet50_imagenet_pretrained_not_finetuned',
    'embedding_dim': 2048,
    'n_images': int(len(embeddings)),
    'n_sessions': int(manifest_reset.session_id.nunique()),
    'n_same_specimen_pairs': int(len(same_df)),
    'n_cross_specimen_pairs': int(len(cross_df)),
    'same_specimen_sim_mean': float(same_sim_all.mean()),
    'same_specimen_sim_median': float(np.median(same_sim_all)),
    'cross_specimen_sim_mean': float(cross_df.cosine_sim.mean()),
    'cross_specimen_sim_median': float(cross_df.cosine_sim.median()),
    'flag_percentile': FLAG_PERCENTILE,
    'flag_threshold_high': threshold_high,
    'n_flagged_cross_specimen': int(len(flagged_cross)),
    'pct_flagged_cross_specimen': 100 * len(flagged_cross) / max(len(cross_df), 1),
    'flag_threshold_low_reverse_check': threshold_low,
    'n_flagged_same_specimen_low_similarity': int(len(flagged_same_low)),
    'threshold_sensitivity': sensitivity_df.to_dict(orient='records'),
    'phash_hamming_mean_same_specimen': float(same_df.phash_hamming.mean()),
    'phash_hamming_mean_cross_specimen': float(cross_df.phash_hamming.mean()),
    'phash_hamming_mean_flagged': float(flagged_cross.phash_hamming.mean()) if len(flagged_cross) else None,
    'n_flagged_near_phash_cutoff': int(len(near_miss)),
    'n_flagged_genuine_phash_blindspot': int(len(genuine_blindspot)),
    'highest_flagged_rate_class': str(hotspot),
}
(OUT_DIR / 'cosine_check_results.json').write_text(json.dumps(summary, indent=2))

md_lines = [
    '# Cosine-similarity specimen cross-check -- numbers for the manuscript', '',
    "- Backbone: ResNet-50, ImageNet-pretrained, NOT fine-tuned on BDLitchi (2048-d embeddings)",
    f"- {summary['n_images']} images embedded across {summary['n_sessions']} sessions",
    f"- {summary['n_same_specimen_pairs']:,} same-specimen pairs, {summary['n_cross_specimen_pairs']:,} cross-specimen pairs (within-session only)",
    f"- Same-specimen cosine sim: mean {summary['same_specimen_sim_mean']:.4f}, median {summary['same_specimen_sim_median']:.4f}",
    f"- Cross-specimen cosine sim: mean {summary['cross_specimen_sim_mean']:.4f}, median {summary['cross_specimen_sim_median']:.4f}",
    f"- Flagging threshold (same-specimen p{FLAG_PERCENTILE}): {threshold_high:.4f}",
    f"- **Cross-specimen pairs flagged as candidate missed duplicates: {summary['n_flagged_cross_specimen']} "
    f"({summary['pct_flagged_cross_specimen']:.3f}% of {summary['n_cross_specimen_pairs']:,} cross-specimen pairs)**",
    f"- Distinct specimens touched by a flagged pair: "
    f"{pd.unique(flagged_cross[['specimen_a','specimen_b']].values.ravel()).shape[0] if len(flagged_cross) else 0}",
    f"- Reverse check -- same-specimen pairs below the cross-specimen p{100-FLAG_PERCENTILE} threshold "
    f"({threshold_low:.4f}): {summary['n_flagged_same_specimen_low_similarity']}",
    f"- pHash Hamming distance, mean: same-specimen {summary['phash_hamming_mean_same_specimen']:.1f}, "
    f"cross-specimen {summary['phash_hamming_mean_cross_specimen']:.1f}, "
    f"flagged {(summary['phash_hamming_mean_flagged'] if summary['phash_hamming_mean_flagged'] is not None else float('nan')):.1f}",
    f"- Of the flagged pairs, {summary['n_flagged_near_phash_cutoff']} were within 3 of the pHash clustering cutoff "
    f"(near-misses) and {summary['n_flagged_genuine_phash_blindspot']} were clear pHash blind spots "
    f"(visually similar, very different hash)",
    f"- Threshold sensitivity: {sensitivity_df.to_dict(orient='records')}",
    f"- Class with the highest flagged rate: {hotspot}",
    '',
    'Next step: open `flagged_cross_specimen_pairs.csv` and the contact-sheet figure and '
    'visually confirm whether the top few flagged pairs are genuinely the same physical leaf. '
    'That human judgement call is what turns this from a number into a defensible claim.',
]
(OUT_DIR / 'manuscript_numbers_cosine.md').write_text('\n'.join(md_lines))
print('\n'.join(md_lines))


## What to bring back

From `/kaggle/working/revision_cosine/`, download and bring back:
- `manuscript_numbers_cosine.md` — the summary numbers, ready to paste into a message.
- `flagged_cross_specimen_pairs.csv` — inspect the top rows yourself first; you know these
  leaves better than any model.
- `figures/fig_cosine_flagged_contact_sheet.png` — the visual side-by-side of the most
  suspicious pairs. This is the fastest way to judge whether the flagged pairs are real.
- `figures/fig_cosine_same_vs_cross_specimen.png` and `table_cosine_specimen_check.tex`
  — the distribution figure and main summary table.
- `figures/fig_cosine_vs_phash_hamming.png` — shows whether flagged pairs are pHash
  near-misses or genuine blind spots; this is what determines which way the paper reads.
- `table_cosine_threshold_sensitivity.tex` and `threshold_sensitivity.csv` — shows the
  result isn't an artifact of picking the 5th percentile specifically.
- `table_cosine_flagged_breakdown.tex` and `flagged_breakdown_by_class.csv` — whether
  flagged pairs concentrate in the four single-session classes already named as
  highest-risk in the manuscript.
- `table_cosine_example_pairs.tex` — a ready-to-drop-in LaTeX table of concrete examples.

Once you have these, send them over and the manuscript text (Methods, and either a new
paragraph in the specimen-construction section or a Limitations update) can be drafted
to match whichever of the two outcomes actually happened.
